In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# cmfe(기존 LTAU 기반)와 cue4unc(MC Dropout / Deep Ensemble)의 결과를 함께 비교한다.
# 세 결과 모두 동일한 CUE 아키텍처 + 동일한 5-fold 내부 CV 파이프라인에서 나온 것이며,
# 유일한 차이는 modality 불확실성을 계산하는 방법(u_m(x))이다:
#   - cmfe:            LTAU (loss-trajectory 기반 k-NN 불확실성)
#   - cue4unc/MC_Dropout:  MC Dropout (T회 확률적 forward pass의 예측 분산)
#   - cue4unc/Deep_Ensemble: Deep Ensemble (서로 다른 seed로 학습한 독립 모델들의 예측 분산)
# scaffold_results: DeepChem ScaffoldSplitter로 나눈 5-fold(8_Kd_scaffold_split.ipynb /
# 8_Ki_scaffold_split.ipynb, selectivity_df/scaffold_{Kd,Ki})로 학습한 결과를 비교한다
# (run_cue_scaffold_all_*.sh / run_cue_scaffold_mc_dropout.sh / run_cue_scaffold_ensemble.sh 참고).
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/scaffold_results'
cue4unc_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue4unc/scaffold_results'

datasets = ['DIS', 'INH']
# cue4unc의 run_cmfe_mc_dropout.sh / run_cmfe_ensemble.sh가 학습하는 타겟과 동일하게 맞춘다.
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 세 가지 소스: (표시 이름, metrics.csv 루트 경로)
# scaffold 학습 스크립트(run_cue_scaffold_mc_dropout.sh)는 VERSION="v1"만 생성하므로
# (v2/v3 하이퍼파라미터 버전은 scaffold split으로는 아직 돌리지 않음) MC Dropout도 v1을 사용한다.
model_sources = {
    'LTAU': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
    'MC Dropout': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v1', 'CUE'),
    'Deep Ensemble': os.path.join(cue4unc_base, 'Deep_Ensemble', '{dataset}', '{target}', 'v1', 'CUE'),
}

# 그래프에 표시될 모델 순서
model_order = ['MC Dropout', 'Deep Ensemble', 'LTAU']

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cue/cue4unc의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    cmfe와 cue4unc 모두 동일한 aggregate_metrics 함수를 사용하므로 스키마가 동일하다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")

✅ 데이터 로드 완료: 24개의 항목이 수집되었습니다.


In [ ]:
DATASET_FILENAMES = {'DIS': 'DIS', 'INH': 'INH'}
METRIC_FILENAMES = {'rmse': 'RMSE', 'r2': 'R2', 'pcc': 'PCC'}
TARGET_COLORS = {
    'PI': '#ef476f',
    'Ssel': '#ffd166',
    'WS2': '#06d6a0',
    'RS2': '#118ab2',
}


def plot_stacked_comparison(df_all, dataset, metric_name, output_dir, model_order=model_order, targets=targets):
    """Model(y축)별로 targets(PI/Ssel/WS2/RS2)의 metric 값을 누적해서 쌓은 가로 막대그래프를 그린다.

    막대 하나 = 모델 하나, 세그먼트 하나 = target 하나의 metric 값이고,
    막대 끝에는 4개 target 값의 합(누적값)을 텍스트로 표시한다.
    x축은 0부터 시작해서 모든 target segment가 항상 온전히 보이도록 한다.
    legend/축 라벨/모델명은 전부 숨기고, y축을 뒤집어서 model_order의 마지막 항목(LTAU)이
    가장 아래로 오도록 한다. top/right spine을 제거해서 합계 숫자가 박스 안에 갇힌 것처럼
    보이지 않도록 한다.

    r2처럼 중간 target 값이 음수인 경우, 누적 위치가 최종 합계(total)보다 오른쪽으로
    더 튀어나갔다가 되돌아올 수 있다(예: RS2의 r2가 음수). 이때 라벨을 total 위치에
    찍으면 실제로 그려진 막대 안쪽(이전 segment 영역)에 박히므로, 스택 도중 도달하는
    가장 오른쪽 지점(max_extent)을 따로 추적해서 라벨/축 범위를 여기 기준으로 잡는다.
    """
    df_sub = df_all[df_all['Dataset'] == dataset]

    y = np.arange(len(model_order))
    segment_values = {}
    for target in targets:
        values = []
        for model in model_order:
            row = df_sub[(df_sub['Model'] == model) & (df_sub['Target'] == target)]
            v = row[metric_name].values[0] if (not row.empty and metric_name in row.columns) else 0.0
            values.append(v)
        segment_values[target] = np.array(values, dtype=float)

    totals = sum(segment_values.values())

    cumulative = np.zeros(len(model_order))
    max_extent = np.zeros(len(model_order))
    for target in targets:
        cumulative = cumulative + segment_values[target]
        max_extent = np.maximum(max_extent, cumulative)

    x_max = max_extent.max()
    margin = x_max * 0.12 if x_max != 0 else 0.1

    fig, ax = plt.subplots(figsize=(8, 5))
    left = np.zeros(len(model_order))

    for target in targets:
        values = segment_values[target]
        ax.barh(y, values, left=left, color=TARGET_COLORS.get(target, '#cccccc'),
                edgecolor='black', linewidth=1, label=target)
        left += values

    # 막대가 실제로 그려진 가장 오른쪽 지점(max_extent) 바로 뒤에 누적 합계(sum) 표시
    for i, total in enumerate(totals):
        ax.text(max_extent[i] + margin * 0.15, y[i], f'{total:.3f}',
                va='center', ha='left', fontsize=12, fontweight='bold')

    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.tick_params(axis='y', which='both', length=0)
    ax.set_ylabel('')
    ax.set_xlabel('')
    ax.set_xlim(0, x_max + margin)
    ax.invert_yaxis()  # model_order 마지막 항목(LTAU)이 가장 아래로 오도록
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    fname = f"{DATASET_FILENAMES[dataset]}_{METRIC_FILENAMES[metric_name]}.png"
    save_path = os.path.join(output_dir, fname)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'unc_comparison_v2')
os.makedirs(plot_base_dir, exist_ok=True)

for d_name in datasets:
    for m in metrics_of_interest:
        plot_stacked_comparison(df_total, d_name, m, plot_base_dir)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/unc_comparison_v2/DIS_RMSE.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/unc_comparison_v2/DIS_R2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/unc_comparison_v2/DIS_PCC.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/unc_comparison_v2/INH_RMSE.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/unc_comparison_v2/INH_R2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/unc_comparison_v2/INH_PCC.png
